Student ID - 2025AC05069

Name - Rajat Mani Tripathi

ML Assignment 2

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef

# 1. Create model directory to store all .pkl files
os.makedirs('model', exist_ok=True)

print("Fetching Default of Credit Card Clients Dataset...")
# Fetches a dataset with 23 features and 30,000 instances, satisfying the requirements
dataset = fetch_openml(name='default-of-credit-card-clients', version=1, as_frame=True, parser='auto')
X = dataset.data
y = dataset.target.astype(int)

# 2. Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Save only the test data for Streamlit to comply with free-tier bandwidth limits
test_data = X_test.copy()
test_data['Target'] = y_test
test_data.to_csv('test_data.csv', index=False)
print("Saved test_data.csv successfully.")

# 4. Feature Scaling (Critical for Logistic Regression and KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler so the Streamlit app can use it on user inputs
joblib.dump(scaler, 'model/scaler.pkl')

# 5. Initialize the 5 required classification models
# Note: Random Forest is constrained to prevent overfitting and massive file sizes
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest (Ensemble)": RandomForestClassifier(n_estimators=60, max_depth=12, random_state=42, n_jobs=-1)
}

# 6. Train, evaluate, and save each model
results = []
print("\n--- Model Evaluation Metrics ---")

for name, model in models.items():
    # Train the model
    model.fit(X_train_scaled, y_train)

    # Generate predictions
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, "predict_proba") else [0]*len(y_test)

    # Calculate the 6 mandatory metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        "ML Model Name": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

    # Format the file name correctly
    file_name = name.replace(" ", "_").replace("(", "").replace(")", "").lower()

    # Save the trained model file with compression level 3 to bypass the 25 MB limit
    joblib.dump(model, f'model/{file_name}.pkl', compress=3)

print("All models trained and saved successfully (Files are compressed to bypass GitHub limits)!\n")

# 7. Display final table to copy exactly into the README.md
results_df = pd.DataFrame(results)
print("Final Comparison Table (For the README.md):")
print(results_df.to_string(index=False))

Fetching Default of Credit Card Clients Dataset...
Saved test_data.csv successfully.

--- Model Evaluation Metrics ---
All models trained and saved successfully (Files are compressed to bypass GitHub limits)!

Final Comparison Table (For your README.md):
           ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
     Logistic Regression    0.8077 0.7076     0.6868  0.2396 0.3553 0.3244
           Decision Tree    0.7152 0.6079     0.3704  0.4115 0.3899 0.2052
                     kNN    0.7928 0.7014     0.5487  0.3564 0.4322 0.3233
             Naive Bayes    0.7525 0.7249     0.4515  0.5539 0.4975 0.3386
Random Forest (Ensemble)    0.8158 0.7718     0.6550  0.3534 0.4591 0.3848
